# 00 - Unión de datasets

Este notebook unifica los datasets originales de Netflix para películas y series en una sola tabla analítica.

Objetivo:
- cargar los CSV raw separados
- normalizar columnas y tipos
- concatenar la información en un único catálogo
- guardar la base unificada en data/processed


In [3]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / 'data' / 'raw').exists():
        project_root = candidate
        break

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(exist_ok=True, parents=True)

movies_path = raw_dir / 'netflix_movies_detailed_up_to_2025.csv'
tv_path = raw_dir / 'netflix_tv_shows_detailed_up_to_2025.csv'

movies_df = pd.read_csv(movies_path, low_memory=False)
tv_df = pd.read_csv(tv_path, low_memory=False)

print(f'Películas raw: {len(movies_df):,}')
print(f'Series raw: {len(tv_df):,}')
print(f'Columnas películas: {movies_df.columns.tolist()}')
print(f'Columnas series: {tv_df.columns.tolist()}')

all_columns = list(movies_df.columns) + [col for col in tv_df.columns if col not in movies_df.columns]
movies_df = movies_df.reindex(columns=all_columns)
tv_df = tv_df.reindex(columns=all_columns)

catalog_df = pd.concat([movies_df, tv_df], ignore_index=True, sort=False)

# Normalización simple de tipo para evitar inconsistencias durante el análisis
for column in ['date_added', 'country', 'genres', 'language', 'rating', 'duration', 'description']:
    if column in catalog_df.columns:
        catalog_df[column] = catalog_df[column].astype('object')

output_path = processed_dir / 'catalogo_streamview.csv'
catalog_df.to_csv(output_path, index=False)

print(f'Base unificada exportada en: {output_path}')
print(f'Filas finales: {len(catalog_df):,}')
print(f'Columnas finales: {list(catalog_df.columns)}')
print(catalog_df.head(2).to_string(index=False))


Películas raw: 16,000
Series raw: 16,000
Columnas películas: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
Columnas series: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average']
Base unificada exportada en: C:\Users\admin\Documents\GitHub\visualizacion-de-datos-StreamView-Analytics\data\processed\catalogo_streamview.csv
Filas finales: 32,000
Columnas finales: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
 show_id  type               title          director                                                                 

## Conclusión de la fase 0: preparación del catálogo base

Hallazgos clave:
- Los datos originales están separados en dos fuentes: películas y series de Netflix.
- Ambos archivos comparten la misma semántica de negocio, pero no la misma estructura exacta de columnas, por lo que se requiere alineación antes del análisis.
- La unión mantiene el contexto completo del catálogo y genera una base única de 32.000 registros, lista para exploración, limpieza y modelado.
- La salida final queda publicada en data/processed/catalogo_streamview.csv como la fuente maestra del resto del proyecto.
